# E3.2 · Governing autonomy rather than approving tools

**Function E — AI Governance for Agentic Systems → Running the Programme — the CISO Office**  ·  *Security of AI*

Builds on **[E3.1 · Translating agentic risk upward](https://spbreed.github.io/cyber-commons/lessons/E3.1.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Write the delegated-authority policy.

**Why a security engineer needs it.** A per-tool review queue becomes a bottleneck and then a bypass. The control it builds is: a policy on delegated authority instead of tool-by-tool approval.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An approved-tools list grows until it is a list of everything, at which point it governs nothing. Autonomy levels and conditions still work at a thousand agents, because they attach to behaviour rather than to product names.

> **At CyberTravels.** CyberTravels will not stop at four agents. An approved-tools list stops governing at about forty; autonomy levels with conditions attach to behaviour and keep working.

## 2 · The framework

```
   autonomy levels, with conditions attached

   L0 proposes only            no conditions
   L1 acts, reversible         logged, sampled
   L2 acts, irreversible       approval + budget + egress control
   L3 acts, no human in loop   L2 + continuous verification + stop authority

   attaches to behaviour, so it still works at a thousand agents
```

Tool-approval processes do not scale for agents. The list of tools grows weekly,
each request needs context the approver does not have, and the queue becomes a
rubber stamp within a quarter.

**Govern autonomy instead.** The autonomy ladder from A1.1 is the right unit
because it is stable — there will always be four rungs — and because it maps
directly onto what can go wrong:

| Rung | Governance |
|---|---|
| **L1** | Self-service. Register it. No further review. |
| **L2** | Register + named owner. Approval gate on every writer, enforced by policy. |
| **L2.5** | Risk tier + blast-radius budget + drift monitoring + tested stop. |
| **L3** | All of L2.5, plus held-out evaluation per release and board sign-off. |

Two properties make this work: a request can be evaluated in minutes by checking
the manifest against the rung, and the policy does not need rewriting when a new
tool appears.

The rung that decides your programme's fate is **L1**. If L1 requires approval,
nobody registers anything and your inventory dies.

## 3 · The procedure, as a skill

Governing the rung rather than the tool. The skill computes each request's blast radius, derives the rung it supports, and refuses with the condition attached — because a bare refusal produces an appeal and a conditional one produces a fix.

### The skill — [`skills/programme/autonomy-ladder-decisions/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/programme/autonomy-ladder-decisions/SKILL.md)

```yaml
name: autonomy-ladder-decisions
description: >-
  Approve or refuse an autonomy request against the rung its blast radius and
  gating actually support, rather than approving the tool it wants. Use when
  agents are being promoted to act unattended.
allowed-tools: Read, Grep, Glob
```

# Govern the rung, not the tool

Approving tools one at a time produces an agent nobody approved: each tool was
reasonable, and the combination acts unattended on production. Governing the
**rung** asks a different question — what does this agent's blast radius and
gating support — and the answer is computed rather than negotiated.

## When to use this

Every request to raise an agent's autonomy, and as a periodic re-check, because
tools accumulate.

## Procedure

**1 — Publish the ladder.** L1 suggests, L2 acts in a sandbox, L2.5 acts with
approval on irreversible actions, L3 acts unattended. With, for each rung, the
governance and the budget it requires.

**2 — Compute the request's blast radius.** Reachable resources weighted by
scope, with gated actions discounted. Ungated writers are what usually decide
the outcome.

**3 — Derive the supported rung from the radius and the gating,** not from the
requester's ask. Then compare. The gap is the conversation.

**4 — Refuse with the condition attached.** "Refused at L2 for ungated writers"
tells the requester what to change. A bare refusal produces an appeal; a
conditional one produces a pull request.

**5 — Re-evaluate on tool change.** A rung approved with three tools does not
carry over to five. Make the tool manifest the trigger for re-evaluation.

## Output contract

```json
{
  "ladder": [{"rung": "str", "governance": "str", "budget": "str"}],
  "requests": [{"name": "str", "asked": "str", "tools": ["str"], "gated": ["str"],
                "blast": 0, "supported": "str", "verdict": "approved|refused", "condition": "str|null"}],
  "reevaluate_on": ["tool added", "scope widened", "model changed"]
}
```

## Failure modes

- **Approving tools.** The combination is what acts.
- **A bare refusal.** Nobody knows what to fix.
- **No re-evaluation trigger.** The manifest grows and the rung does not move.

In [ ]:
# The code is not in this notebook. It is the file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/programme/autonomy-ladder-decisions/scripts/autonomy_ladder_decisions.py
SCRIPT = "skills/programme/autonomy-ladder-decisions/scripts/autonomy_ladder_decisions.py"

import glob, os, subprocess, sys

# The skills tree: the attached dataset on Kaggle, the checkout locally.
_ROOTS = sorted(glob.glob("/kaggle/input/**/cyber-commons-skills", recursive=True)) + [".", "..", "../.."]
_root = next((r for r in _ROOTS if os.path.isfile(os.path.join(r, SCRIPT))), None)
if _root is None:
    raise SystemExit("skills tree not found. On Kaggle add the dataset "
                     "cybercommons/cyber-commons-skills; locally run from a checkout.")

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The four rungs print with their governance and budgets. The doc-summariser is approved at L1; `triage-bot` is refused at L2 for ungated writers; the refund agent is refused at L2.5 for exceeding the budget and approved once gated. Tool approval scales to 240 reviews a month at 120 agents against 5 hours for rung governance, and a committee-gated L1 leaves roughly 108 shadow assets.

## Your turn

Write your own per-rung policy in four lines and check what L1 costs a team today. If registering a read-only copilot needs an approval, your inventory is already incomplete and you cannot see by how much.

---

**Next → [E3.3 · Sequencing the programme](https://spbreed.github.io/cyber-commons/lessons/E3.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E3.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E3.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*